<a href="https://colab.research.google.com/github/talhanoor23/Deep-Learning-Experiments/blob/main/Regression/NeuralNetwork_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
print(tf.__version__)

2.17.1


# **Data_Preprocessing**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib
import plotly.express as px
import os
%matplotlib inline

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d yasserh/housing-prices-dataset

Dataset URL: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset
License(s): CC0-1.0
  0% 0.00/4.63k [00:00<?, ?B/s]
100% 4.63k/4.63k [00:00<00:00, 9.32MB/s]


In [ ]:
!unzip /content/housing-prices-dataset.zip

Archive:  /content/housing-prices-dataset.zip
  inflating: Housing.csv             


In [ ]:
house_df = pd.read_csv('Housing.csv')

In [ ]:
house_df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [ ]:
house_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [ ]:
house_df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [ ]:
house_df.isna().sum()

,0
price,0
area,0
bedrooms,0
bathrooms,0
stories,0
mainroad,0
guestroom,0
basement,0
hotwaterheating,0
airconditioning,0


In [ ]:
fig = px.histogram(house_df, x="price", color='basement', marginal='box', hover_data=house_df.columns)
fig.update_layout(title_text='Housing Prices Dataset', bargap=0.1)

In [ ]:
fig = px.histogram(house_df, x="price", color='mainroad', marginal='box', hover_data=house_df.columns)
fig.update_layout(title_text='Housing Prices Dataset', bargap=0.1)

In [ ]:
fig = px.histogram(house_df, x="airconditioning", color='hotwaterheating', marginal='box', hover_data=house_df.columns)
fig.update_layout(title_text='Housing Prices Dataset', bargap=0.1)

In [ ]:
house_df.basement.value_counts()

,count
basement,
no,354
yes,191


In [ ]:
house_df.airconditioning.value_counts()

,count
airconditioning,
no,373
yes,172


In [ ]:
house_df.bedrooms.value_counts()

,count
bedrooms,
3,300
2,136
4,95
5,10
6,2
1,2


In [ ]:
house_df.stories.value_counts()

,count
stories,
2,238
1,227
4,41
3,39


In [ ]:
fig = px.scatter(house_df, x="price", y="area", color='airconditioning', hover_data=['furnishingstatus'])
fig.update_traces(marker_size = 5)#furnishing,basement, area has major effect on prices.

In [ ]:
house_df.price.corr(house_df.area)

0.5359973457780796

In [ ]:
house_df.price.corr(house_df.stories)

0.4207123661886163

In [ ]:
house_df.price.corr(house_df.parking)

0.3843936486357259

In [ ]:
categ_numeric = {'no':0, 'yes':1}
road_numeric = house_df.mainroad.map(categ_numeric)
house_df.price.corr(road_numeric)

0.2968984892639764

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(house_df, test_size=0.2, random_state=42)

In [ ]:
train_inputs = train_set.drop('price', axis=1)
train_targets = train_set['price'].copy()
test_inputs = test_set.drop('price', axis=1)
test_targets = test_set['price'].copy()

In [ ]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
scaler.fit(train_inputs[numeric_cols])

StandardScaler()

In [ ]:
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

In [ ]:
train_inputs.describe()

,area,bedrooms,bathrooms,stories,parking
count,4.360000e+02,4.360000e+02,4.360000e+02,4.360000e+02,4.360000e+02
mean,1.222264e-16,1.629685e-16,7.333583e-17,6.926162e-17,-6.518741e-17
std,1.001149e+00,1.001149e+00,1.001149e+00,1.001149e+00,1.001149e+00
min,-1.591502e+00,-2.622298e+00,-5.579503e-01,-9.124989e-01,-8.030587e-01
25%,-7.058568e-01,-1.283514e+00,-5.579503e-01,-9.124989e-01,-8.030587e-01
50%,-2.970974e-01,5.527092e-02,-5.579503e-01,2.542152e-01,-8.030587e-01
75%,5.476719e-01,5.527092e-02,1.539173e+00,2.542152e-01,3.679566e-01
max,5.016775e+00,4.071624e+00,5.733420e+00,2.587644e+00,2.709987e+00


In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()

In [ ]:
encoder.fit(train_inputs[categorical_cols])

OneHotEncoder()

In [ ]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

In [ ]:
train_inputs[encoded_cols] = encoder.transform(train_set[categorical_cols]).toarray()
test_inputs[encoded_cols] = encoder.transform(test_set[categorical_cols]).toarray()

In [ ]:
train_inputs.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,...,basement_yes,hotwaterheating_no,hotwaterheating_yes,airconditioning_no,airconditioning_yes,prefarea_no,prefarea_yes,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
46,0.384168,0.055271,1.539173,2.587644,yes,no,no,no,yes,0.367957,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
93,0.929181,0.055271,1.539173,-0.912499,yes,no,yes,no,yes,2.709987,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
335,-0.607755,-1.283514,-0.557950,-0.912499,yes,no,yes,no,yes,1.538972,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
412,-1.155492,0.055271,-0.557950,0.254215,yes,no,yes,no,no,-0.803059,...,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
471,-0.637730,0.055271,-0.557950,0.254215,yes,no,no,no,no,-0.803059,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
train_inputs = train_inputs[numeric_cols + encoded_cols]
test_inputs = test_inputs[numeric_cols + encoded_cols]

# **Model_Creation_Training**

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(256, activation='relu', input_shape=(train_inputs.shape[1],)))
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(32, activation='relu'))
model.add(tf.keras.layers.Dense(1))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
              loss='mae',
              metrics=['mae'])

In [ ]:
history = model.fit(train_inputs, train_targets,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=32)

Epoch 1/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 4358720.0000 - mae: 4358720.0000 - val_loss: 3473163.0000 - val_mae: 3473163.0000
Epoch 2/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2896826.5000 - mae: 2896826.5000 - val_loss: 1252062.7500 - val_mae: 1252062.7500
Epoch 3/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1371341.6250 - mae: 1371341.6250 - val_loss: 1035794.3750 - val_mae: 1035794.3750
Epoch 4/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 923209.6250 - mae: 923209.6250 - val_loss: 863598.3750 - val_mae: 863598.3750
Epoch 5/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 788763.1875 - mae: 788763.1875 - val_loss: 813833.2500 - val_mae: 813833.2500
Epoch 6/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 819667.5000 - mae: 819667.5000 - val_loss: 822171.3750 - val_mae: 822171.3750
Epoch 7/100
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 750105.8750 - mae: 750105.8750 - val_loss: 822038.7500 - val_mae: 822038.7500
Epoch 8/100
11/11 ━━━━━━━━━━

In [ ]:
model.evaluate(test_inputs, test_targets)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 963869.9375 - mae: 963870.4375 


[1007154.5625, 1007155.0625]